# Grupo 1 · Capa RAW · Rubén
Pipeline E12 → E13 (Ruta Agroambiental)

Este notebook implementa la capa **RAW** del pipeline de datos:
1. **Ingesta y verificación de fuentes**: Comprobación estricta de tamaño en bytes y huella criptográfica SHA-256 de las fuentes crudas (`eva.csv`, `divipola.csv`, `agrosavia.csv`) contra `kit/fuentes.json`.
2. **Control de calidad y reglas de rechazo**: Identificación de duplicados exactos, validaciones de dominio (códigos DANE, rangos temporales, áreas y producciones no negativas, rangos de pH) y separación de registros no conformes en `rechazados.parquet`.
3. **Tipificación y normalización**: Estandarización de nombres y generación de `clave_territorial` mediante `norm()`, conversión numérica (`float64`, `int64`) y formato de fechas (`date`).
4. **Almacenamiento analítico Parquet**: Escritura de `eva.parquet`, `divipola.parquet`, `agrosavia.parquet` y `rechazados.parquet`.
5. **Auditoría, controles y antiduplicidad**: Generación de `manifiesto_entradas.json`, `controles_raw.json` y registro transversal en `kit/salidas/grupo_1/registro_raw.jsonl` para prevenir reescrituras innecesarias.


## 1. Entorno, rutas y configuración
Se detecta la raíz del repositorio localmente buscando `kit/fuentes.json`, se genera un identificador único de ejecución (`run_id` de 12 caracteres hexadecimales) y se parametrizan las carpetas de entrada y salida, asegurando que las salidas se almacenen en `kit/salidas/grupo_1/<run_id>/raw/` fuera del control de versiones.


In [ ]:
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from uuid import uuid4
import hashlib
import json
import re
import unicodedata
import platform

import duckdb
import pandas as pd
import pyarrow
import pyarrow.parquet as pq
from IPython.display import display


def encontrar_repositorio(inicio=None):
    """Localiza la raíz del repositorio big-data-postgraduate buscando kit/fuentes.json."""
    inicio = Path.cwd() if inicio is None else Path(inicio)
    for candidato in (inicio, *inicio.parents):
        if (candidato / 'kit' / 'fuentes.json').is_file():
            return candidato
        anidado = candidato / 'big-data-postgraduate'
        if (anidado / 'kit' / 'fuentes.json').is_file():
            return anidado
    raise FileNotFoundError('No se encontró la raíz de big-data-postgraduate')


REPOSITORIO = encontrar_repositorio()

# Celda de parámetros (convención del grupo)
RUN_ID = None          # None = generar uno nuevo
if RUN_ID is None:
    RUN_ID = uuid4().hex[:12]

ENTRADA_DIR = REPOSITORIO / 'kit' / 'data' / 'raw'
SALIDA_DIR = REPOSITORIO / 'kit' / 'salidas' / 'grupo_1' / RUN_ID / 'raw'
REGISTRO_RUTA = REPOSITORIO / 'kit' / 'salidas' / 'grupo_1' / 'registro_raw.jsonl'
FUENTES_RUTA = REPOSITORIO / 'kit' / 'fuentes.json'
FORZAR = False         # True = reprocesar aunque ya exista una ejecución válida

print(f"Repositorio: {REPOSITORIO}")
print(f"Run ID:      {RUN_ID}")
print(f"Entrada:     {ENTRADA_DIR}")
print(f"Salida:      {SALIDA_DIR}")
print(f"Forzar:      {FORZAR}")
print(f"Entorno:     Python {platform.python_version()} | pandas {pd.__version__} | duckdb {duckdb.__version__} | pyarrow {pyarrow.__version__}")


## 2. Funciones de control, normalización y manifiesto de fuentes
Se implementan las funciones utilitarias necesarias para:
- `sha256_archivo`: Cálculo de huella SHA-256 por bloques sin saturar memoria.
- `firma_dataframe`: Obtención de hash determinístico del contenido ordenado.
- `agregar_motivo`: Acumulación de múltiples causales de rechazo separadas por `|`.
- `norm`: Función idéntica a `kit/talleres.py` (línea 7) para normalización de entidades territoriales.

Se contrastan los archivos locales contra las huellas y tamaños esperados en `kit/fuentes.json` y se comprueba el **Control 1** (`coincide == True`).


In [ ]:
def sha256_archivo(ruta, bloque=1024 * 1024):
    """Calcula la huella SHA-256 por bloques sin cargar archivos completos en memoria."""
    firma = hashlib.sha256()
    with Path(ruta).open('rb') as archivo:
        for fragmento in iter(lambda: archivo.read(bloque), b''):
            firma.update(fragmento)
    return firma.hexdigest()


def firma_dataframe(datos, columnas, orden):
    """Calcula una huella SHA-256 determinística del DataFrame normalizado y ordenado."""
    normalizado = datos.loc[:, columnas].copy()
    normalizado = normalizado.sort_values(orden, kind='stable').reset_index(drop=True)
    normalizado = normalizado.astype('string').fillna('<NA>')
    texto = normalizado.to_csv(index=False, lineterminator='\n')
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()


def agregar_motivo(serie, mascara, motivo):
    """Agrega un motivo de rechazo concatenando con '|' si ya había motivos previos."""
    previo = serie.loc[mascara]
    serie.loc[mascara] = previo.where(previo.eq(''), previo + '|') + motivo


def norm(x):
    """Normaliza texto: quita acentos, pasa a mayúsculas y deja solo letras, dígitos y espacios simples."""
    x = ''.join(c for c in unicodedata.normalize('NFKD', str(x)) if not unicodedata.combining(c))
    return ' '.join(re.sub(r'[^A-Z0-9 ]', ' ', x.upper()).split())


In [ ]:
with FUENTES_RUTA.open(encoding='utf-8') as f:
    catalogo_fuentes = {item['id']: item for item in json.load(f)['fuentes']}

fuentes_orden = ['eva', 'divipola', 'agrosavia']
manifiesto_entradas = []
shas_archivos = []
datos_crudos = {}

for fid in fuentes_orden:
    meta = catalogo_fuentes[fid]
    ruta_archivo = ENTRADA_DIR / meta['archivo']
    assert ruta_archivo.is_file(), f"No existe el archivo de entrada: {ruta_archivo}"

    bytes_reales = ruta_archivo.stat().st_size
    bytes_esperados = meta['bytes']
    sha_real = sha256_archivo(ruta_archivo)
    sha_esperado = meta['sha256']
    coincide = (bytes_reales == bytes_esperados) and (sha_real == sha_esperado)

    # Lectura como texto para preservar ceros a la izquierda
    df_crudo = pd.read_csv(ruta_archivo, dtype=str, keep_default_na=False, encoding='utf-8-sig')
    datos_crudos[fid] = df_crudo

    item = {
        'id': fid,
        'ruta': str(ruta_archivo.relative_to(REPOSITORIO)).replace('\\', '/'),
        'bytes': bytes_reales,
        'bytes_esperados': bytes_esperados,
        'sha256': sha_real,
        'sha256_esperado': sha_esperado,
        'coincide': coincide,
        'filas_leidas': len(df_crudo),
        'columnas': df_crudo.columns.tolist()
    }
    manifiesto_entradas.append(item)
    shas_archivos.append(sha_real)

    # Control 1: Para cada fuente, coincide == True en el manifiesto
    assert coincide, f"Control 1 fallido para fuente {fid}: tamaño o hash SHA-256 no coinciden con fuentes.json"

firma_entrada = hashlib.sha256(''.join(shas_archivos).encode('utf-8')).hexdigest()
print(f"Firma de entrada conjunta (SHA-256): {firma_entrada}")
display(pd.DataFrame(manifiesto_entradas)[['id', 'bytes', 'bytes_esperados', 'sha256', 'coincide', 'filas_leidas']])


## 3. Ingesta y validación de EVA (Evaluaciones Agropecuarias)
Se procesan las evaluaciones agropecuarias municipales. Se validan las reglas de rechazo:
- `DUPLICADO_EXACTO`: Fila idéntica a otra anterior en todas las columnas.
- `MUNICIPIO_INVALIDO`: Código DANE que tras `zfill(5)` no posee exactamente 5 dígitos.
- `ANIO_INVALIDO`: Año no numérico o vacío.
- `AREA_INVALIDA`: Área cosechada vacía, no numérica o negativa (0 es válido).
- `PRODUCCION_INVALIDA`: Producción vacía, no numérica o negativa.

Las filas aceptadas se tipifican rigurosamente conforme al contrato de datos.


In [ ]:
df_eva_raw = datos_crudos['eva']
df_eva = df_eva_raw.copy()
df_eva['fila_fuente'] = (df_eva.index + 2).astype('int64')

cols_eval_eva = df_eva_raw.columns.tolist()
motivos_eva = pd.Series('', index=df_eva.index, dtype='string')

# Reglas de rechazo EVA
dup_eva = df_eva.duplicated(subset=cols_eval_eva, keep='first')
agregar_motivo(motivos_eva, dup_eva, 'DUPLICADO_EXACTO')

cod_mpio_eva_zfill = df_eva['Código Dane municipio'].astype(str).str.strip().str.zfill(5)
mpio_eva_invalido = ~cod_mpio_eva_zfill.str.fullmatch(r'\d{5}', na=False)
agregar_motivo(motivos_eva, mpio_eva_invalido, 'MUNICIPIO_INVALIDO')

anio_str = df_eva['Año'].astype(str).str.strip()
anio_num = pd.to_numeric(anio_str, errors='coerce')
anio_invalido = anio_str.eq('') | anio_num.isna() | anio_num.mod(1).ne(0)
agregar_motivo(motivos_eva, anio_invalido, 'ANIO_INVALIDO')

area_str = df_eva['Área cosechada'].astype(str).str.strip().str.replace(',', '.')
area_num = pd.to_numeric(area_str, errors='coerce')
area_invalida = area_str.eq('') | area_num.isna() | (area_num < 0)
agregar_motivo(motivos_eva, area_invalida, 'AREA_INVALIDA')

prod_str = df_eva['Producción'].astype(str).str.strip().str.replace(',', '.')
prod_num = pd.to_numeric(prod_str, errors='coerce')
prod_invalida = prod_str.eq('') | prod_num.isna() | (prod_num < 0)
agregar_motivo(motivos_eva, prod_invalida, 'PRODUCCION_INVALIDA')

es_rechazado_eva = motivos_eva.ne('')

# Construir rechazados de EVA
rechazados_eva_list = []
for idx in df_eva.index[es_rechazado_eva]:
    row = df_eva.loc[idx]
    orig_dict = {col: row[col] for col in cols_eval_eva}
    rechazados_eva_list.append({
        'fuente': 'eva',
        'fila_fuente': int(row['fila_fuente']),
        'motivos': str(motivos_eva.loc[idx]),
        'registro_original': json.dumps(orig_dict, ensure_ascii=False)
    })
df_rechazados_eva = pd.DataFrame(rechazados_eva_list)
if df_rechazados_eva.empty:
    df_rechazados_eva = pd.DataFrame(columns=['fuente', 'fila_fuente', 'motivos', 'registro_original'])
    df_rechazados_eva['fila_fuente'] = df_rechazados_eva['fila_fuente'].astype('int64')

# Construir aceptados de EVA
aceptados_eva = df_eva.loc[~es_rechazado_eva]
df_eva_parquet = pd.DataFrame({
    'codigo_departamento': aceptados_eva['Código Dane departamento'].astype(str).str.strip().str.zfill(2).astype('string'),
    'codigo_municipio': cod_mpio_eva_zfill.loc[~es_rechazado_eva].astype('string'),
    'departamento': aceptados_eva['Departamento'].astype(str).str.strip().astype('string'),
    'municipio': aceptados_eva['Municipio'].astype(str).str.strip().astype('string'),
    'grupo_cultivo': aceptados_eva['Grupo cultivo'].astype(str).str.strip().astype('string'),
    'cultivo': aceptados_eva['Cultivo'].astype(str).str.strip().astype('string'),
    'anio': anio_num.loc[~es_rechazado_eva].astype('int64'),
    'periodo': aceptados_eva['Periodo'].astype(str).str.strip().astype('string'),
    'area_sembrada_ha': pd.to_numeric(aceptados_eva['Área sembrada'].astype(str).str.strip().str.replace(',', '.'), errors='coerce').astype('float64'),
    'area_cosechada_ha': area_num.loc[~es_rechazado_eva].astype('float64'),
    'produccion_t': prod_num.loc[~es_rechazado_eva].astype('float64'),
    'rendimiento_publicado_t_ha': pd.to_numeric(aceptados_eva['Rendimiento'].astype(str).str.strip().str.replace(',', '.'), errors='coerce').astype('float64'),
    'ciclo': aceptados_eva['Ciclo del cultivo'].astype(str).str.strip().astype('string'),
    'estado_fisico': aceptados_eva['Estado físico del cultivo'].astype(str).str.strip().astype('string'),
    'codigo_cultivo': aceptados_eva['Código del cultivo'].astype(str).str.strip().astype('string'),
    'fila_fuente': aceptados_eva['fila_fuente'].astype('int64')
})

motivos_eva_conteo = {
    str(k): int(v) for k, v in motivos_eva[es_rechazado_eva].str.split('|').explode().value_counts().items()
} if es_rechazado_eva.any() else {}

print(f"EVA - Leídas: {len(df_eva_raw):,} | Aceptadas: {len(df_eva_parquet):,} | Rechazadas: {len(df_rechazados_eva):,}")
if motivos_eva_conteo:
    print("Motivos de rechazo EVA:", motivos_eva_conteo)
display(df_eva_parquet.head(3))


## 4. Ingesta y normalización de DIVIPOLA
DIVIPOLA contiene la codificación político-administrativa de Colombia.
- Se formatean los códigos DANE a dos (`codigo_departamento`) y cinco (`codigo_municipio`) caracteres con ceros a la izquierda.
- Se convierte `longitud` y `latitud` reemplazando comas por puntos a `float64`.
- Se genera la columna `clave_territorial` calculada como `norm(departamento) + '|' + norm(municipio)`.
- Se valida el **Control 6**: unicidad obligatoria de `codigo_municipio` y `clave_territorial`.


In [ ]:
df_div_raw = datos_crudos['divipola']
df_div = df_div_raw.copy()
df_div['fila_fuente'] = (df_div.index + 2).astype('int64')

cod_dpto_div_zfill = df_div['Código Departamento'].astype(str).str.strip().str.zfill(2)
cod_mpio_div_zfill = df_div['Código Municipio'].astype(str).str.strip().str.zfill(5)

dpto_div_norm = df_div['Nombre Departamento'].astype(str).apply(norm)
mpio_div_norm = df_div['Nombre Municipio'].astype(str).apply(norm)
claves_territoriales_div = dpto_div_norm + '|' + mpio_div_norm

# Control 6: Unicidad territorial en DIVIPOLA
assert cod_mpio_div_zfill.is_unique, "Control 6 fallido: Códigos de municipio repetidos en DIVIPOLA"
assert claves_territoriales_div.is_unique, "Control 6 fallido: Claves territoriales repetidas en DIVIPOLA"

df_divipola_parquet = pd.DataFrame({
    'codigo_departamento': cod_dpto_div_zfill.astype('string'),
    'codigo_municipio': cod_mpio_div_zfill.astype('string'),
    'departamento': df_div['Nombre Departamento'].astype(str).str.strip().astype('string'),
    'municipio': df_div['Nombre Municipio'].astype(str).str.strip().astype('string'),
    'tipo': df_div['Tipo: Municipio / Isla / Área no municipalizada'].astype(str).str.strip().astype('string'),
    'longitud': pd.to_numeric(df_div['longitud'].astype(str).str.strip().str.replace(',', '.'), errors='raise').astype('float64'),
    'latitud': pd.to_numeric(df_div['Latitud'].astype(str).str.strip().str.replace(',', '.'), errors='raise').astype('float64'),
    'clave_territorial': claves_territoriales_div.astype('string')
})

df_rechazados_div = pd.DataFrame(columns=['fuente', 'fila_fuente', 'motivos', 'registro_original'])
df_rechazados_div['fila_fuente'] = df_rechazados_div['fila_fuente'].astype('int64')

print(f"DIVIPOLA - Leídas: {len(df_div_raw):,} | Aceptadas: {len(df_divipola_parquet):,} | Rechazadas: 0")
display(df_divipola_parquet.head(3))


## 5. Ingesta y validación de AGROSAVIA
AGROSAVIA aporta análisis de laboratorio de suelos.
Reglas de validación:
- `DUPLICADO_EXACTO`: Fila idéntica en todas las columnas.
- `FECHA_INVALIDA`: Fecha no analizable bajo `%d/%m/%Y`.
- `PH_FUERA_DOMINIO`: pH numérico fuera del intervalo químico [0, 14].
- `TERRITORIO_VACIO`: Registro con departamento o municipio vacío.

Tratamiento de nulos: los valores `ND`, vacíos o textos no numéricos en variables fisicoquímicas pasan a nulo (`float64`) sin constituir motivo de rechazo. Se extrae `anio` como `int64` y `fecha` como tipo `date`.


In [ ]:
df_agro_raw = datos_crudos['agrosavia']
df_agro = df_agro_raw.copy()
df_agro['fila_fuente'] = (df_agro.index + 2).astype('int64')

cols_eval_agro = df_agro_raw.columns.tolist()
motivos_agro = pd.Series('', index=df_agro.index, dtype='string')

# Reglas de rechazo AGROSAVIA
dup_agro = df_agro.duplicated(subset=cols_eval_agro, keep='first')
agregar_motivo(motivos_agro, dup_agro, 'DUPLICADO_EXACTO')

fecha_dt = pd.to_datetime(df_agro['Fecha de Análisis'].astype(str).str.strip(), format='%d/%m/%Y', errors='coerce')
agregar_motivo(motivos_agro, fecha_dt.isna(), 'FECHA_INVALIDA')

ph_raw = df_agro['pH agua:suelo'].astype(str).str.strip().str.replace(',', '.')
ph_num = pd.to_numeric(ph_raw, errors='coerce')
ph_fuera = ph_num.notna() & ((ph_num < 0) | (ph_num > 14))
agregar_motivo(motivos_agro, ph_fuera, 'PH_FUERA_DOMINIO')

dpto_agro_str = df_agro['Departamento'].astype(str).str.strip()
mpio_agro_str = df_agro['Municipio'].astype(str).str.strip()
terr_vacio = dpto_agro_str.eq('') | mpio_agro_str.eq('')
agregar_motivo(motivos_agro, terr_vacio, 'TERRITORIO_VACIO')

es_rechazado_agro = motivos_agro.ne('')

# Construir rechazados de AGROSAVIA
rechazados_agro_list = []
for idx in df_agro.index[es_rechazado_agro]:
    row = df_agro.loc[idx]
    orig_dict = {col: row[col] for col in cols_eval_agro}
    rechazados_agro_list.append({
        'fuente': 'agrosavia',
        'fila_fuente': int(row['fila_fuente']),
        'motivos': str(motivos_agro.loc[idx]),
        'registro_original': json.dumps(orig_dict, ensure_ascii=False)
    })
df_rechazados_agro = pd.DataFrame(rechazados_agro_list)
if df_rechazados_agro.empty:
    df_rechazados_agro = pd.DataFrame(columns=['fuente', 'fila_fuente', 'motivos', 'registro_original'])
    df_rechazados_agro['fila_fuente'] = df_rechazados_agro['fila_fuente'].astype('int64')

# Construir aceptados de AGROSAVIA
aceptados_agro = df_agro.loc[~es_rechazado_agro]
fecha_valida_agro = fecha_dt.loc[~es_rechazado_agro]
claves_agro = [norm(d) + '|' + norm(m) for d, m in zip(aceptados_agro['Departamento'], aceptados_agro['Municipio'])]

def extraer_num(columna):
    return pd.to_numeric(aceptados_agro[columna].astype(str).str.strip().str.replace(',', '.'), errors='coerce').astype('float64')

df_agrosavia_parquet = pd.DataFrame({
    'id_muestra': aceptados_agro['Secuencial'].astype(str).str.strip().astype('string'),
    'fila_fuente': aceptados_agro['fila_fuente'].astype('int64'),
    'fecha': fecha_valida_agro.dt.date,
    'anio': fecha_valida_agro.dt.year.astype('int64'),
    'departamento': aceptados_agro['Departamento'].astype(str).str.strip().astype('string'),
    'municipio': aceptados_agro['Municipio'].astype(str).str.strip().astype('string'),
    'clave_territorial': pd.Series(claves_agro, index=aceptados_agro.index, dtype='string'),
    'cultivo': aceptados_agro['Cultivo'].astype(str).str.strip().astype('string'),
    'ph': extraer_num('pH agua:suelo'),
    'materia_organica': extraer_num('Materia organica'),
    'fosforo': extraer_num('Fósforo Bray II'),
    'aluminio': extraer_num('Aluminio intercambiable'),
    'calcio': extraer_num('Calcio intercambiable'),
    'magnesio': extraer_num('Magnesio intercambiable'),
    'potasio': extraer_num('Potasio intercambiable'),
    'cic': extraer_num('capacidad de intercambio cationico'),
    'conductividad': extraer_num('Conductividad electrica')
})

motivos_agro_conteo = {
    str(k): int(v) for k, v in motivos_agro[es_rechazado_agro].str.split('|').explode().value_counts().items()
} if es_rechazado_agro.any() else {}

print(f"AGROSAVIA - Leídas: {len(df_agro_raw):,} | Aceptadas: {len(df_agrosavia_parquet):,} | Rechazadas: {len(df_rechazados_agro):,}")
if motivos_agro_conteo:
    print("Motivos de rechazo AGROSAVIA:", motivos_agro_conteo)
display(df_agrosavia_parquet.head(3))


## 6. Escritura de artefactos Parquet, rechazos y controles de calidad
Se genera el directorio de ejecución `SALIDA_DIR` y se escriben los 6 artefactos de salida del contrato.
A continuación, se reexaminan los archivos persistidos con pandas y DuckDB para evaluar de forma exhaustiva los controles formales de calidad (Controles 1 al 7).


In [ ]:
# Consolidación de filas rechazadas
df_rechazados_parquet = pd.concat([df_rechazados_eva, df_rechazados_div, df_rechazados_agro], ignore_index=True)

# Crear directorio exclusivo de la corrida
SALIDA_DIR.mkdir(parents=True, exist_ok=True)

# 1. Manifiesto de entradas
ruta_manifiesto = SALIDA_DIR / 'manifiesto_entradas.json'
with ruta_manifiesto.open('w', encoding='utf-8') as f:
    json.dump(manifiesto_entradas, f, ensure_ascii=False, indent=2)

# 2. Tablas Parquet
ruta_eva_parquet = SALIDA_DIR / 'eva.parquet'
ruta_div_parquet = SALIDA_DIR / 'divipola.parquet'
ruta_agro_parquet = SALIDA_DIR / 'agrosavia.parquet'
ruta_rech_parquet = SALIDA_DIR / 'rechazados.parquet'

df_eva_parquet.to_parquet(ruta_eva_parquet, index=False)
df_divipola_parquet.to_parquet(ruta_div_parquet, index=False)
df_agrosavia_parquet.to_parquet(ruta_agro_parquet, index=False)
df_rechazados_parquet.to_parquet(ruta_rech_parquet, index=False)

# Relectura y cálculo de firmas de contenido
COLUMNAS_EVA = df_eva_parquet.columns.tolist()
COLUMNAS_DIV = df_divipola_parquet.columns.tolist()
COLUMNAS_AGRO = df_agrosavia_parquet.columns.tolist()
COLUMNAS_RECH = df_rechazados_parquet.columns.tolist()

firmas_contenido = {
    'eva': firma_dataframe(df_eva_parquet, COLUMNAS_EVA, ['fila_fuente']),
    'divipola': firma_dataframe(df_divipola_parquet, COLUMNAS_DIV, ['codigo_municipio']),
    'agrosavia': firma_dataframe(df_agrosavia_parquet, COLUMNAS_AGRO, ['fila_fuente'])
}
firma_contenido_combinada = hashlib.sha256(
    f"{firmas_contenido['eva']}|{firmas_contenido['divipola']}|{firmas_contenido['agrosavia']}".encode('utf-8')
).hexdigest()

# 3. Metadatos de control
controles_raw = {
    'run_id': RUN_ID,
    'firma_entrada': firma_entrada,
    'estado': 'valida',
    'fuentes': {
        'eva': {
            'leidas': len(df_eva_raw),
            'aceptadas': len(df_eva_parquet),
            'rechazadas': len(df_rechazados_eva),
            'motivos': motivos_eva_conteo
        },
        'agrosavia': {
            'leidas': len(df_agro_raw),
            'aceptadas': len(df_agrosavia_parquet),
            'rechazadas': len(df_rechazados_agro),
            'motivos': motivos_agro_conteo
        },
        'divipola': {
            'leidas': len(df_div_raw),
            'aceptadas': len(df_divipola_parquet),
            'rechazadas': len(df_rechazados_div),
            'motivos': {}
        }
    },
    'firmas_contenido': firmas_contenido,
    'errores': []
}

ruta_controles = SALIDA_DIR / 'controles_raw.json'
with ruta_controles.open('w', encoding='utf-8') as f:
    json.dump(controles_raw, f, ensure_ascii=False, indent=2)

print(f"Artefactos generados exitosamente en: {SALIDA_DIR}")


In [ ]:
print("=== VERIFICACIÓN FORMAL DE CONTROLES (SECCIÓN 7) ===")

# Control 1: Manifiesto coincide == True
assert all(item['coincide'] for item in manifiesto_entradas), "Control 1 fallido: discrepancias en manifiesto"
print("✓ Control 1: Huellas y tamaños de entrada coinciden con kit/fuentes.json")

# Control 2: leidas == aceptadas + rechazadas
for nombre_fte, fte_data in controles_raw['fuentes'].items():
    assert fte_data['leidas'] == fte_data['aceptadas'] + fte_data['rechazadas'], (
        f"Control 2 fallido en {nombre_fte}: {fte_data['leidas']} != {fte_data['aceptadas']} + {fte_data['rechazadas']}"
    )
print("✓ Control 2: Conciliación total (leídas == aceptadas + rechazadas) en todas las fuentes")

# Control 3: fila_fuente única por fuente en rechazados.parquet
assert not df_rechazados_parquet.duplicated(subset=['fuente', 'fila_fuente']).any(), (
    "Control 3 fallido: Existen filas fuente duplicadas en rechazados.parquet para una misma fuente"
)
print("✓ Control 3: Cada fila rechazada aparece una sola vez en rechazados.parquet")

# Control 4: Columnas y tipos de cada Parquet iguales al contrato
eva_releido = pd.read_parquet(ruta_eva_parquet)
div_releido = pd.read_parquet(ruta_div_parquet)
agro_releido = pd.read_parquet(ruta_agro_parquet)
rech_releido = pd.read_parquet(ruta_rech_parquet)

contratos_esperados = {
    'eva': [
        'codigo_departamento', 'codigo_municipio', 'departamento', 'municipio',
        'grupo_cultivo', 'cultivo', 'anio', 'periodo', 'area_sembrada_ha',
        'area_cosechada_ha', 'produccion_t', 'rendimiento_publicado_t_ha',
        'ciclo', 'estado_fisico', 'codigo_cultivo', 'fila_fuente'
    ],
    'divipola': [
        'codigo_departamento', 'codigo_municipio', 'departamento', 'municipio',
        'tipo', 'longitud', 'latitud', 'clave_territorial'
    ],
    'agrosavia': [
        'id_muestra', 'fila_fuente', 'fecha', 'anio', 'departamento', 'municipio',
        'clave_territorial', 'cultivo', 'ph', 'materia_organica', 'fosforo',
        'aluminio', 'calcio', 'magnesio', 'potasio', 'cic', 'conductividad'
    ],
    'rechazados': ['fuente', 'fila_fuente', 'motivos', 'registro_original']
}

assert eva_releido.columns.tolist() == contratos_esperados['eva'], "Control 4: Columnas EVA no coinciden con contrato"
assert div_releido.columns.tolist() == contratos_esperados['divipola'], "Control 4: Columnas DIVIPOLA no coinciden con contrato"
assert agro_releido.columns.tolist() == contratos_esperados['agrosavia'], "Control 4: Columnas AGROSAVIA no coinciden con contrato"
assert rech_releido.columns.tolist() == contratos_esperados['rechazados'], "Control 4: Columnas RECHAZADOS no coinciden con contrato"
print("✓ Control 4: Columnas y orden en Parquet cumplen estrictamente el contrato")

# Control 5: Códigos DANE con ceros a la izquierda y longitud exacta
assert eva_releido['codigo_departamento'].str.len().eq(2).all() and eva_releido['codigo_departamento'].str.fullmatch(r'\d{2}').all()
assert eva_releido['codigo_municipio'].str.len().eq(5).all() and eva_releido['codigo_municipio'].str.fullmatch(r'\d{5}').all()
assert div_releido['codigo_departamento'].str.len().eq(2).all() and div_releido['codigo_departamento'].str.fullmatch(r'\d{2}').all()
assert div_releido['codigo_municipio'].str.len().eq(5).all() and div_releido['codigo_municipio'].str.fullmatch(r'\d{5}').all()
print("✓ Control 5: Códigos DANE validados: departamento (2 dígitos), municipio (5 dígitos)")

# Control 6: divipola codigo_municipio y clave_territorial únicos
assert div_releido['codigo_municipio'].is_unique, "Control 6: Código municipio no es único en divipola.parquet"
assert div_releido['clave_territorial'].is_unique, "Control 6: Clave territorial no es única en divipola.parquet"
print("✓ Control 6: Unicidad territorial garantizada en divipola.parquet")

# Control 7: Releer Parquets y comprobar mismas filas
assert len(eva_releido) == len(df_eva_parquet), "Control 7: Discrepancia en filas de eva.parquet"
assert len(div_releido) == len(df_divipola_parquet), "Control 7: Discrepancia en filas de divipola.parquet"
assert len(agro_releido) == len(df_agrosavia_parquet), "Control 7: Discrepancia en filas de agrosavia.parquet"
assert len(rech_releido) == len(df_rechazados_parquet), "Control 7: Discrepancia en filas de rechazados.parquet"
print("✓ Control 7: Relectura íntegra y conteos idénticos en todos los Parquet")

# Verificación de integridad adicional con DuckDB
with duckdb.connect() as con:
    cnt_eva = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [str(ruta_eva_parquet)]).fetchone()[0]
    cnt_div = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [str(ruta_div_parquet)]).fetchone()[0]
    cnt_agro = con.execute("SELECT COUNT(*) FROM read_parquet(?)", [str(ruta_agro_parquet)]).fetchone()[0]
    assert cnt_eva == len(eva_releido) and cnt_div == len(div_releido) and cnt_agro == len(agro_releido)
print("✓ Validación analítica con DuckDB exitosa")


## 7. Registro transversal y antiduplicidad
Se actualiza `kit/salidas/grupo_1/registro_raw.jsonl` con el resultado de la corrida (`run_id, fecha, firma_entrada, firma_contenido, estado, ruta_salida`).
Se verifica el **Control 8**:
- Detección de duplicidad con `FORZAR = False`.
- Estabilidad determinística de la `firma_contenido` bajo reprocesamiento.


In [ ]:
registro_linea = {
    'run_id': RUN_ID,
    'fecha': datetime.now(ZoneInfo('America/Bogota')).isoformat(),
    'firma_entrada': firma_entrada,
    'firma_contenido': firma_contenido_combinada,
    'estado': 'valida',
    'ruta_salida': str(SALIDA_DIR.relative_to(REPOSITORIO)).replace('\\', '/')
}

REGISTRO_RUTA.parent.mkdir(parents=True, exist_ok=True)
with REGISTRO_RUTA.open('a', encoding='utf-8') as f:
    f.write(json.dumps(registro_linea, ensure_ascii=False) + '\n')

print(f"Registro asentado en: {REGISTRO_RUTA}")
print("Detalle:", registro_linea)

# Control 8: Verificación de reejecución y estabilidad
print("\n=== VERIFICACIÓN CONTROL 8: ANTIDUPLICIDAD Y ESTABILIDAD ===")

# Prueba 8.1: Comprobar detección previa
registros_actuales = [
    json.loads(linea) for linea in REGISTRO_RUTA.read_text(encoding='utf-8').strip().split('\n') if linea.strip()
]
encontrados = [r for r in registros_actuales if r.get('firma_entrada') == firma_entrada and r.get('estado') == 'valida']
assert len(encontrados) >= 1, "Control 8.1 fallido: No se encontró registro válido en registro_raw.jsonl"
print(f"ya procesado en {encontrados[0]['ruta_salida']}")

# Prueba 8.2: Determinismo de firma_contenido
firmas_recalculadas = {
    'eva': firma_dataframe(df_eva_parquet, COLUMNAS_EVA, ['fila_fuente']),
    'divipola': firma_dataframe(df_divipola_parquet, COLUMNAS_DIV, ['codigo_municipio']),
    'agrosavia': firma_dataframe(df_agrosavia_parquet, COLUMNAS_AGRO, ['fila_fuente'])
}
firma_contenido_reproducida = hashlib.sha256(
    f"{firmas_recalculadas['eva']}|{firmas_recalculadas['divipola']}|{firmas_recalculadas['agrosavia']}".encode('utf-8')
).hexdigest()

assert firma_contenido_reproducida == firma_contenido_combinada, "Control 8.2 fallido: Las firmas de contenido difieren"
print(f"✓ Control 8 superado: Antiduplicidad comprobada y firma_contenido estable: {firma_contenido_combinada}")


## 8. Resumen de ejecución y criterios de entrega
Consolidación final de métricas de la capa RAW:
- Conteo de registros por fuente.
- Los 3 motivos de rechazo más frecuentes.
- Verificación física de los 6 artefactos de salida del contrato.


In [ ]:
resumen_fuentes = []
for fid in fuentes_orden:
    info = controles_raw['fuentes'][fid]
    resumen_fuentes.append({
        'Fuente': fid.upper(),
        'Filas Leídas': info['leidas'],
        'Filas Aceptadas': info['aceptadas'],
        'Filas Rechazadas': info['rechazadas'],
        'Tasa Rechazo (%)': round((info['rechazadas'] / info['leidas']) * 100, 4) if info['leidas'] > 0 else 0.0
    })

df_resumen = pd.DataFrame(resumen_fuentes)
display(df_resumen)

todos_motivos = {}
for fid, fte_data in controles_raw['fuentes'].items():
    for mot, cnt in fte_data['motivos'].items():
        todos_motivos[mot] = todos_motivos.get(mot, 0) + cnt

top3_motivos = sorted(todos_motivos.items(), key=lambda x: x[1], reverse=True)[:3]
print("\n--- TOP 3 MOTIVOS DE RECHAZO MÁS FRECUENTES ---")
if top3_motivos:
    for puesto, (mot, cantidad) in enumerate(top3_motivos, 1):
        print(f"{puesto}. {mot}: {cantidad:,} registros")
else:
    print("No se presentaron registros rechazados.")

archivos_esperados = [
    'manifiesto_entradas.json',
    'eva.parquet',
    'divipola.parquet',
    'agrosavia.parquet',
    'rechazados.parquet',
    'controles_raw.json'
]
print("\n--- VERIFICACIÓN DE ARTEFACTOS GENERADOS ---")
for arch in archivos_esperados:
    ruta_arch = SALIDA_DIR / arch
    assert ruta_arch.is_file(), f"Falta archivo requerido: {arch}"
    tam_kb = ruta_arch.stat().st_size / 1024
    print(f"✓ {arch:25s} ({tam_kb:,.1f} KB)")

print("\n🚀 Entrega Capa RAW completada satisfactoriamente para la etapa SILVER.")
